In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split

In [2]:
df = pd.read_csv("../data/processed_reviews.csv")

y = df["sentiment"]

tf = TfidfVectorizer(max_features=10000)
X = tf.fit_transform(df["review"])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [3]:
X_train = X_train.toarray()
X_test = X_test.toarray()

train_set = TensorDataset(
    torch.from_numpy(X_train).float(),
    torch.from_numpy(y_train.values).float()
)

test_set = TensorDataset(
    torch.from_numpy(X_test).float(),
    torch.from_numpy(y_test.values).float()
)

train_loader = DataLoader(
    train_set,
    shuffle=True,
    batch_size=64
)

test_loader = DataLoader(
    test_set,
    shuffle=False,
    batch_size=64
)

In [8]:
train_set[0]

(tensor([0.0000, 0.0000, 0.2077,  ..., 0.0000, 0.0000, 0.0000]), tensor(0.))

In [4]:
class RNN(nn.Module):
    def __init__(self, input_size, hidden_size=128, num_layers=1):
        super().__init__()

        self.hidden_size = hidden_size
        self.num_layers = num_layers

        # RNN Layer
        self.rnn = nn.RNN(input_size, hidden_size, num_layers, batch_first=True)

        # Fully connected layer
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        # optional
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size)
        
        output, _ = self.rnn(x, h0)

        output = self.fc(output[:, -1, :])
        return output

In [5]:
input_size = X_train.shape[1]

model = RNN(input_size)

criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters())

### Training the RNN

In [6]:
epochs = 20

for epoch in range(epochs):
    model.train()

    for Xb, yb in train_loader:
        optimizer.zero_grad()

        Xb = Xb.unsqueeze(1)

        outputs = model(Xb)
        outputs = torch.sigmoid(outputs.squeeze())

        loss = criterion(outputs, yb)
        loss.backward()
        optimizer.step()

    print(f"epoch = {epoch + 1}/{epochs} and loss = {loss.item()}")

epoch = 1/20 and loss = 0.38131290674209595
epoch = 2/20 and loss = 0.18178653717041016
epoch = 3/20 and loss = 0.1417294293642044
epoch = 4/20 and loss = 0.07992894947528839
epoch = 5/20 and loss = 0.10253355652093887
epoch = 6/20 and loss = 0.27204400300979614
epoch = 7/20 and loss = 0.052830856293439865
epoch = 8/20 and loss = 0.04407264292240143
epoch = 9/20 and loss = 0.07571496069431305
epoch = 10/20 and loss = 0.21116995811462402
epoch = 11/20 and loss = 0.1532057821750641
epoch = 12/20 and loss = 0.05618848651647568
epoch = 13/20 and loss = 0.09091266989707947
epoch = 14/20 and loss = 0.1209733635187149
epoch = 15/20 and loss = 0.11807101219892502
epoch = 16/20 and loss = 0.13936184346675873
epoch = 17/20 and loss = 0.3555448055267334
epoch = 18/20 and loss = 0.06793969869613647
epoch = 19/20 and loss = 0.06240817904472351
epoch = 20/20 and loss = 0.0676569864153862


 ### Evaluation

In [7]:
model.eval()

with torch.no_grad():
    correct_vals = 0
    tot_vals = 0
    
    for Xb, yb in test_loader:
        Xb = Xb.unsqueeze(1)

        outputs = model(Xb)
        predicted = (torch.sigmoid(outputs.squeeze()) > 0.5).float()

        tot_vals += yb.size(0)
        correct_vals += (predicted == yb).sum().item()

    print(f"accuracy = {correct_vals / tot_vals * 100}")

accuracy = 84.1585156801452
